# Retrieval

Run `pipeline.vector_store.search` for one question string and one source file.
Set `SHOW_TEXT = True` to print chunk text. The last cell deletes this collection.


In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent
os.chdir(ROOT)
for entry in (ROOT / "notebooks", ROOT):
    text = str(entry)
    if text not in sys.path:
        sys.path.insert(0, text)

from helpers import accepted_pdfs, new_session_id, preview, timed

from pipeline.embedder import embed_document
from pipeline.extractor import extract

SHOW_TEXT = False
pdf_dir = (
    Path(os.environ["NOTEBOOK_PDF_DIR"])
    if os.environ.get("NOTEBOOK_PDF_DIR")
    else ROOT / "notebooks" / "inputs"
)
session_id = new_session_id()
documents = []

with timed("setup") as setup:
    accepted, rejected = accepted_pdfs(pdf_dir)
    for path in accepted:
        try:
            document = await extract(path)
        except Exception as exc:
            print(path.name, "failure", type(exc).__name__, exc)
            continue
        if document.metadata.total_chunks == 0:
            print(path.name, "failure", "RuntimeError", "produced no text")
            continue
        await embed_document(document, session_id)
        documents.append(document)

print(session_id)
print(f"setup: {setup['seconds']:.3f}s")
print(f"ready: {len(documents)}  rejected: {len(rejected)}")

In [ ]:
source = documents[0].metadata.filename if documents else ""
question = "Copy the requested value from the document."
n_results = 5
print(source)
print(question)
print(n_results)

In [ ]:
from pipeline.vector_store import search

with timed("search") as search_time:
    hits = search(question, session_id, source, n_results) if source else []

print(f"search: {search_time['seconds']:.3f}s")

In [ ]:
print("rank\tpage\theadings\tdistance\tchars")
for rank, hit in enumerate(hits, start=1):
    headings = " > ".join(hit["headings"])
    print(
        rank,
        hit["page"],
        headings,
        f"{hit['distance']:.4f}",
        len(hit["text"]),
        sep="\t",
    )
    if SHOW_TEXT:
        print(preview(hit["text"], SHOW_TEXT))

In [ ]:
from pipeline.embedder import delete_collection

delete_collection(session_id)
print(session_id, "deleted")